In [2]:
# ============================================================
# VQA 5개 모델 확률 검증 + 점수 기반 Soft-voting 앙상블
#
# 입력 폴더:
# MyDrive/ensemble/
#
# 필요한 파일:
# - qwen35_27b_test_probabilities_03_final_other.csv
# - qwen38_27b_test_probabilities_03_final_other.csv
# - qwen35_4b_test_probabilities_03_final_other.csv
# - gemma_e2b_test_probabilities_03_final_other.csv
# - gemma_31b_test_probabilities_03_final_other.csv
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Google Drive 안전 마운트
# ============================================================

DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive")

if DEFAULT_DRIVE_ROOT.is_dir():
    DRIVE_ROOT = DEFAULT_DRIVE_ROOT
    print(f"기존 Google Drive 사용: {DRIVE_ROOT}")

else:
    from google.colab import drive

    mount_index = 0

    while True:
        if mount_index == 0:
            mount_point = Path("/content/gdrive_mount")
        else:
            mount_point = Path(
                f"/content/gdrive_mount_{mount_index}"
            )

        if not mount_point.exists():
            mount_point.mkdir(parents=True)
            break

        if mount_point.is_dir() and not any(mount_point.iterdir()):
            break

        mount_index += 1

    drive.mount(
        str(mount_point),
        force_remount=False,
    )

    DRIVE_ROOT = mount_point / "MyDrive"

print(f"DRIVE_ROOT: {DRIVE_ROOT}")


# ============================================================
# 2. 입출력 경로
# ============================================================

ENSEMBLE_ROOT = DRIVE_ROOT / "Ensemble"

OUTPUT_ROOT = (
    ENSEMBLE_ROOT
    / "outputs_score_tuned"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_FILES = {
    "qwen38_27b": (
        ENSEMBLE_ROOT
        / "qwen38_27b_test_probabilities_03_final_other.csv"
    ),

    "qwen35_27b": (
        ENSEMBLE_ROOT
        / "qwen35_27b_test_probabilities_03_final_other.csv"
    ),

    "qwen35_4b": (
        ENSEMBLE_ROOT
        / "qwen35_4b_test_probabilities_03_final_other.csv"
    ),

    "gemma4_31b": (
        ENSEMBLE_ROOT
        / "gemma_31b_test_probabilities_03_final_other.csv"
    ),

    "gemma4_e2b": (
        ENSEMBLE_ROOT
        / "gemma_e2b_test_probabilities_03_final_other.csv"
    ),
}

LABELS = np.array(["a", "b", "c", "d"])
PROBABILITY_COLUMNS = ["p_a", "p_b", "p_c", "p_d"]
EXPECTED_ROW_COUNT = 6714

print(f"입력 폴더: {ENSEMBLE_ROOT}")
print(f"출력 폴더: {OUTPUT_ROOT}")


# ============================================================
# 3. 참고용 단일 모델 공개 점수
# ============================================================

INDIVIDUAL_SCORES = {
    "gemma4_31b": 0.96008,
    "qwen38_27b": 0.95918,
    "qwen35_27b": 0.94489,
    "qwen35_4b": 0.85224,
    "gemma4_e2b": 0.64462,
}

print()
print("=" * 80)
print("단일 모델 점수")
print("=" * 80)

for model_name, score in sorted(
    INDIVIDUAL_SCORES.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{model_name:<12}: {score:.5f}")


# ============================================================
# 4. 입력 파일 존재 검사
# ============================================================

print()
print("=" * 80)
print("입력 파일 확인")
print("=" * 80)

missing_files = {}

for model_name, path in MODEL_FILES.items():
    if path.is_file():
        print(f"[FOUND]   {model_name}")
        print(f"          {path}")
    else:
        print(f"[MISSING] {model_name}")
        print(f"          {path}")

        missing_files[model_name] = path

if missing_files:
    missing_message = "\n".join(
        f"- {model_name}: {path}"
        for model_name, path in missing_files.items()
    )

    raise FileNotFoundError(
        "\n다음 파일을 찾을 수 없습니다.\n"
        f"{missing_message}\n\n"
        "파일명과 MyDrive/ensemble 폴더를 확인하세요."
    )


# ============================================================
# 5. 확률 CSV 로딩 및 검증
# ============================================================

PROBABILITY_COLUMN_SCHEMES = [
    ["p_a", "p_b", "p_c", "p_d"],
    ["a", "b", "c", "d"],
    ["prob_a", "prob_b", "prob_c", "prob_d"],
    ["score_a", "score_b", "score_c", "score_d"],
]


def load_probability_csv(model_name, path):
    frame = pd.read_csv(path)

    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    if "id" not in frame.columns:
        raise ValueError(
            f"[{model_name}] id 열이 없습니다.\n"
            f"파일: {path}\n"
            f"열: {frame.columns.tolist()}"
        )

    detected_columns = None

    for scheme in PROBABILITY_COLUMN_SCHEMES:
        if all(column in frame.columns for column in scheme):
            detected_columns = scheme
            break

    if detected_columns is None:
        raise ValueError(
            f"[{model_name}] 확률 열을 찾지 못했습니다.\n"
            f"파일: {path}\n"
            f"열: {frame.columns.tolist()}"
        )

    frame = frame.rename(
        columns=dict(
            zip(
                detected_columns,
                PROBABILITY_COLUMNS,
            )
        )
    )

    frame["id"] = (
        frame["id"]
        .astype(str)
        .str.strip()
    )

    if frame["id"].eq("").any():
        raise ValueError(
            f"[{model_name}] 빈 ID가 있습니다."
        )

    duplicate_mask = frame["id"].duplicated(
        keep=False
    )

    if duplicate_mask.any():
        duplicate_ids = (
            frame.loc[duplicate_mask, "id"]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"[{model_name}] 중복 ID가 있습니다.\n"
            f"예시: {duplicate_ids[:10]}"
        )

    if len(frame) != EXPECTED_ROW_COUNT:
        raise ValueError(
            f"[{model_name}] 행 수가 잘못되었습니다.\n"
            f"예상: {EXPECTED_ROW_COUNT:,}\n"
            f"실제: {len(frame):,}\n"
            "partial 파일인지 확인하세요."
        )

    for column in PROBABILITY_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    probabilities = frame[
        PROBABILITY_COLUMNS
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(probabilities).all():
        invalid_rows = np.where(
            ~np.isfinite(probabilities).all(axis=1)
        )[0]

        raise ValueError(
            f"[{model_name}] NaN 또는 무한대가 있습니다.\n"
            f"문제 행: {invalid_rows[:10].tolist()}"
        )

    if (probabilities < -1e-8).any():
        invalid_rows = np.where(
            (probabilities < -1e-8).any(axis=1)
        )[0]

        raise ValueError(
            f"[{model_name}] 음수 확률이 있습니다.\n"
            f"문제 행: {invalid_rows[:10].tolist()}"
        )

    probabilities = np.clip(
        probabilities,
        0.0,
        None,
    )

    probability_sums = probabilities.sum(axis=1)

    if (probability_sums <= 0).any():
        invalid_rows = np.where(
            probability_sums <= 0
        )[0]

        raise ValueError(
            f"[{model_name}] 확률 합이 0인 행이 있습니다.\n"
            f"문제 행: {invalid_rows[:10].tolist()}"
        )

    # 각 행의 확률 합을 1로 정규화
    probabilities = (
        probabilities
        / probability_sums[:, None]
    )

    normalized = pd.DataFrame({
        "id": frame["id"].to_numpy(),
        "p_a": probabilities[:, 0],
        "p_b": probabilities[:, 1],
        "p_c": probabilities[:, 2],
        "p_d": probabilities[:, 3],
    })

    calculated_predictions = LABELS[
        np.argmax(
            probabilities,
            axis=1,
        )
    ]

    saved_prediction_column = None

    if "pred" in frame.columns:
        saved_prediction_column = "pred"
    elif "answer" in frame.columns:
        saved_prediction_column = "answer"

    if saved_prediction_column is not None:
        saved_predictions = (
            frame[saved_prediction_column]
            .astype(str)
            .str.strip()
            .str.lower()
            .to_numpy()
        )

        mismatch_count = int(
            np.sum(
                saved_predictions
                != calculated_predictions
            )
        )

        if mismatch_count > 0:
            print(
                f"[참고] {model_name}: "
                f"저장된 {saved_prediction_column}와 "
                f"확률 argmax가 {mismatch_count:,}개 다릅니다."
            )

    print(
        f"[LOADED] {model_name:<12} "
        f"rows={len(normalized):,}, "
        f"probability_columns={detected_columns}"
    )

    return normalized


# ============================================================
# 6. 모든 모델 로드
# ============================================================

print()
print("=" * 80)
print("확률 CSV 검증")
print("=" * 80)

model_frames = {}

for model_name, path in MODEL_FILES.items():
    model_frames[model_name] = load_probability_csv(
        model_name=model_name,
        path=path,
    )


# ============================================================
# 7. ID 일치 검사 및 정렬
# ============================================================

REFERENCE_MODEL = "qwen38_27b"

reference_frame = model_frames[REFERENCE_MODEL]
reference_ids = reference_frame["id"].tolist()
reference_id_set = set(reference_ids)
row_count = len(reference_ids)

aligned_probabilities = {}

for model_name, frame in model_frames.items():
    current_id_set = set(frame["id"])

    missing_ids = (
        reference_id_set
        - current_id_set
    )

    extra_ids = (
        current_id_set
        - reference_id_set
    )

    if missing_ids or extra_ids:
        raise ValueError(
            f"[{model_name}] ID 구성이 다릅니다.\n"
            f"누락 ID: {len(missing_ids):,}개\n"
            f"추가 ID: {len(extra_ids):,}개\n"
            f"누락 예시: {sorted(missing_ids)[:10]}\n"
            f"추가 예시: {sorted(extra_ids)[:10]}"
        )

    aligned_frame = (
        frame
        .set_index("id")
        .loc[reference_ids]
    )

    aligned_probabilities[model_name] = (
        aligned_frame[PROBABILITY_COLUMNS]
        .to_numpy(dtype=np.float64)
    )

print()
print(
    f"ID 검증 완료: 모든 모델이 "
    f"{row_count:,}개의 동일한 ID를 가지고 있습니다."
)


# ============================================================
# 8. 점수 기반 조정 가중치
#
# current_best_top3:
#   기존 공개 점수 0.96127
#
# score_tuned_balanced:
#   보수적인 조정
#
# score_tuned_main:
#   가장 먼저 제출할 추천 후보
#
# score_tuned_gemma_heavy:
#   Gemma31B를 더 강조한 공격적인 후보
#
# score_tuned_with_qwen4b:
#   Qwen4B 다양성을 2%만 반영한 실험 후보
# ============================================================

WEIGHT_SETS = {
    "current_best_top3": {
        "qwen38_27b": 0.40,
        "gemma4_31b": 0.35,
        "qwen35_27b": 0.25,
    },

    "score_tuned_balanced": {
        "gemma4_31b": 0.40,
        "qwen38_27b": 0.40,
        "qwen35_27b": 0.20,
    },

    "score_tuned_main": {
        "gemma4_31b": 0.42,
        "qwen38_27b": 0.40,
        "qwen35_27b": 0.18,
    },

    "score_tuned_gemma_heavy": {
        "gemma4_31b": 0.46,
        "qwen38_27b": 0.39,
        "qwen35_27b": 0.15,
    },

    "score_tuned_with_qwen4b": {
        "gemma4_31b": 0.41,
        "qwen38_27b": 0.39,
        "qwen35_27b": 0.18,
        "qwen35_4b": 0.02,
    },
}


# ============================================================
# 9. 앙상블 생성 함수
# ============================================================

def create_ensemble(
    ensemble_name,
    weights,
):
    unavailable_models = [
        model_name
        for model_name in weights
        if model_name not in aligned_probabilities
    ]

    if unavailable_models:
        raise KeyError(
            f"[{ensemble_name}] 사용할 수 없는 모델: "
            f"{unavailable_models}"
        )

    total_weight = float(
        sum(weights.values())
    )

    if total_weight <= 0:
        raise ValueError(
            f"[{ensemble_name}] 가중치 합이 0 이하입니다."
        )

    normalized_weights = {
        model_name: weight / total_weight
        for model_name, weight in weights.items()
    }

    ensemble_probabilities = np.zeros(
        (row_count, 4),
        dtype=np.float64,
    )

    for model_name, weight in normalized_weights.items():
        ensemble_probabilities += (
            weight
            * aligned_probabilities[model_name]
        )

    ensemble_probabilities /= (
        ensemble_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    ensemble_predictions = LABELS[
        np.argmax(
            ensemble_probabilities,
            axis=1,
        )
    ]

    # --------------------------------------------
    # Kaggle 제출 파일
    # --------------------------------------------

    submission = pd.DataFrame({
        "id": reference_ids,
        "answer": ensemble_predictions,
    })

    submission_path = (
        OUTPUT_ROOT
        / f"submission_{ensemble_name}.csv"
    )

    submission.to_csv(
        submission_path,
        index=False,
        encoding="utf-8",
    )

    # --------------------------------------------
    # 진단 정보
    # --------------------------------------------

    sorted_probabilities = np.sort(
        ensemble_probabilities,
        axis=1,
    )

    margins = (
        sorted_probabilities[:, -1]
        - sorted_probabilities[:, -2]
    )

    entropies = -np.sum(
        ensemble_probabilities
        * np.log(
            np.clip(
                ensemble_probabilities,
                1e-12,
                1.0,
            )
        ),
        axis=1,
    )

    diagnostics = pd.DataFrame({
        "id": reference_ids,
        "answer": ensemble_predictions,
        "p_a": ensemble_probabilities[:, 0],
        "p_b": ensemble_probabilities[:, 1],
        "p_c": ensemble_probabilities[:, 2],
        "p_d": ensemble_probabilities[:, 3],
        "confidence": ensemble_probabilities.max(axis=1),
        "margin": margins,
        "entropy": entropies,
    })

    model_prediction_list = []

    for model_name in normalized_weights:
        model_predictions = LABELS[
            np.argmax(
                aligned_probabilities[model_name],
                axis=1,
            )
        ]

        diagnostics[
            f"pred_{model_name}"
        ] = model_predictions

        diagnostics[
            f"confidence_{model_name}"
        ] = aligned_probabilities[
            model_name
        ].max(axis=1)

        model_prediction_list.append(
            model_predictions
        )

    prediction_matrix = np.column_stack(
        model_prediction_list
    )

    diagnostics["agreement_count"] = np.sum(
        prediction_matrix
        == ensemble_predictions[:, None],
        axis=1,
    )

    diagnostics[
        "unique_prediction_count"
    ] = np.array([
        len(set(row))
        for row in prediction_matrix
    ])

    diagnostics_path = (
        OUTPUT_ROOT
        / f"diagnostics_{ensemble_name}.csv"
    )

    diagnostics.to_csv(
        diagnostics_path,
        index=False,
        encoding="utf-8",
    )

    print()
    print("-" * 80)
    print(f"앙상블 완료: {ensemble_name}")
    print("-" * 80)

    for model_name, weight in normalized_weights.items():
        print(
            f"{model_name:<12}: "
            f"{weight:.4f}"
        )

    print(f"제출 파일: {submission_path}")
    print(f"진단 파일: {diagnostics_path}")

    return {
        "submission": submission,
        "diagnostics": diagnostics,
        "probabilities": ensemble_probabilities,
        "weights": normalized_weights,
        "submission_path": submission_path,
        "diagnostics_path": diagnostics_path,
    }


# ============================================================
# 10. 모든 가중치 후보 실행
# ============================================================

ensemble_results = {}

for ensemble_name, weights in WEIGHT_SETS.items():
    ensemble_results[ensemble_name] = create_ensemble(
        ensemble_name=ensemble_name,
        weights=weights,
    )


# ============================================================
# 11. 기존 최고 앙상블과 예측 차이 비교
# ============================================================

baseline_name = "current_best_top3"

baseline_answers = (
    ensemble_results[baseline_name]
    ["submission"]["answer"]
    .to_numpy()
)

prediction_comparison = pd.DataFrame({
    "id": reference_ids,
    f"answer_{baseline_name}": baseline_answers,
})

comparison_summary_rows = []

for ensemble_name, result in ensemble_results.items():
    answers = (
        result["submission"]["answer"]
        .to_numpy()
    )

    prediction_comparison[
        f"answer_{ensemble_name}"
    ] = answers

    changed_mask = answers != baseline_answers

    prediction_comparison[
        f"changed_{ensemble_name}"
    ] = changed_mask

    comparison_summary_rows.append({
        "ensemble_name": ensemble_name,
        "changed_vs_current_best": int(
            changed_mask.sum()
        ),
        "changed_ratio": float(
            changed_mask.mean()
        ),
    })

comparison_summary = pd.DataFrame(
    comparison_summary_rows
)

prediction_comparison_path = (
    OUTPUT_ROOT
    / "prediction_comparison.csv"
)

comparison_summary_path = (
    OUTPUT_ROOT
    / "ensemble_comparison_summary.csv"
)

prediction_comparison.to_csv(
    prediction_comparison_path,
    index=False,
    encoding="utf-8",
)

comparison_summary.to_csv(
    comparison_summary_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 12. 가중치 정보 저장
# ============================================================

weight_rows = []

for ensemble_name, result in ensemble_results.items():
    row = {
        "ensemble_name": ensemble_name,
    }

    for model_name in MODEL_FILES:
        row[f"weight_{model_name}"] = (
            result["weights"].get(
                model_name,
                0.0,
            )
        )

    weight_rows.append(row)

weight_summary = pd.DataFrame(
    weight_rows
)

weight_summary_path = (
    OUTPUT_ROOT
    / "ensemble_weight_summary.csv"
)

weight_summary.to_csv(
    weight_summary_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 13. 최종 제출 파일 검증
# ============================================================

for ensemble_name, result in ensemble_results.items():
    submission = result["submission"]

    if len(submission) != EXPECTED_ROW_COUNT:
        raise RuntimeError(
            f"[{ensemble_name}] 행 수가 잘못되었습니다."
        )

    if submission["id"].duplicated().any():
        raise RuntimeError(
            f"[{ensemble_name}] 중복 ID가 있습니다."
        )

    invalid_answers = (
        set(submission["answer"])
        - set(LABELS)
    )

    if invalid_answers:
        raise RuntimeError(
            f"[{ensemble_name}] 잘못된 answer가 있습니다: "
            f"{invalid_answers}"
        )


# ============================================================
# 14. 최종 결과 출력
# ============================================================

print()
print("=" * 80)
print("전체 앙상블 완료")
print("=" * 80)

print(f"전체 샘플 수: {row_count:,}")
print(f"결과 저장 폴더: {OUTPUT_ROOT}")

print()
print("기존 최고 앙상블 대비 변경 수:")

display(
    comparison_summary.sort_values(
        "changed_vs_current_best"
    )
)

print()
print("제출 우선순위:")

submission_priority = [
    "score_tuned_main",
    "score_tuned_balanced",
    "score_tuned_gemma_heavy",
    "score_tuned_with_qwen4b",
    "current_best_top3",
]

for rank, ensemble_name in enumerate(
    submission_priority,
    start=1,
):
    path = ensemble_results[
        ensemble_name
    ]["submission_path"]

    print(f"{rank}. {path}")

print()
print("첫 번째 추천 제출 파일:")
print(
    ensemble_results[
        "score_tuned_main"
    ]["submission_path"]
)

print()
print("추천 제출 파일 미리보기:")

display(
    ensemble_results[
        "score_tuned_main"
    ]["submission"].head(10)
)

Mounted at /content/gdrive_mount_1
DRIVE_ROOT: /content/gdrive_mount_1/MyDrive
입력 폴더: /content/gdrive_mount_1/MyDrive/Ensemble
출력 폴더: /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned

단일 모델 점수
gemma4_31b  : 0.96008
qwen38_27b  : 0.95918
qwen35_27b  : 0.94489
qwen35_4b   : 0.85224
gemma4_e2b  : 0.64462

입력 파일 확인
[FOUND]   qwen38_27b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen38_27b_test_probabilities_03_final_other.csv
[FOUND]   qwen35_27b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen35_27b_test_probabilities_03_final_other.csv
[FOUND]   qwen35_4b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen35_4b_test_probabilities_03_final_other.csv
[FOUND]   gemma4_31b
          /content/gdrive_mount_1/MyDrive/Ensemble/gemma_31b_test_probabilities_03_final_other.csv
[FOUND]   gemma4_e2b
          /content/gdrive_mount_1/MyDrive/Ensemble/gemma_e2b_test_probabilities_03_final_other.csv

확률 CSV 검증
[참고] qwen38_27b: 저장된 pred와 확률 argmax가 5개 다릅니다.
[LOADED] qwen38_

,ensemble_name,changed_vs_current_best,changed_ratio
0,current_best_top3,0,0.000000
1,score_tuned_balanced,31,0.004617
4,score_tuned_with_qwen4b,38,0.005660
2,score_tuned_main,39,0.005809
3,score_tuned_gemma_heavy,72,0.010724



제출 우선순위:
1. /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_score_tuned_main.csv
2. /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_score_tuned_balanced.csv
3. /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_score_tuned_gemma_heavy.csv
4. /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_score_tuned_with_qwen4b.csv
5. /content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_current_best_top3.csv

첫 번째 추천 제출 파일:
/content/gdrive_mount_1/MyDrive/Ensemble/outputs_score_tuned/submission_score_tuned_main.csv

추천 제출 파일 미리보기:


,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,c
5,test_0006.jpg,d
6,test_0007.jpg,b
7,test_0008.jpg,c
8,test_0009.jpg,c
9,test_0010.jpg,a
